# Deformable registration of lung 4DCT with SwinUNETR, TTO and DICOM / RTSTRUCT export

The code lives in the `registration/` package of this repository (`pip install -e .` from the repository root); this notebook only contains configuration and execution. Method and pipeline: `docs/method.md` and `docs/pipeline.md`; recorded outputs of each section: `results/`.

**Data.** Sections 3-8 use the public Learn2Reg lung CT data (`data/LungCT/`, see the README). Sections 9 and 10 use the 4D-Lung case study: subject `107_HM10395` of the public TCIA collection 4D-Lung (CC BY 3.0; planning CT, a later CT and CBCT, RTSTRUCTs; see `data/README.md`). Its images are named after `CLINICAL_CASE`, set in the first code cell. Each section writes its printed output to the matching file of `results/`.

| Module | Content |
|---|---|
| `dataset.py` | `Learn2RegDataset` (with disk cache), `InvertedPair`, `copy_to_local` |
| `augmentation.py` | `RegistrationAugmentation`: flip, rigid motion and intensity transforms on the GPU |
| `model.py` | SwinUNETR + velocity head, efficient window attention, activation checkpointing, scaling & squaring, spatial transformer |
| `losses.py` | NCC, MIND, regularization, Dice, landmark/keypoint losses |
| `metrics.py` | TRE, Dice, Jacobian |
| `training.py` | `PhaseConfig`, `train_phase` (curriculum, resume, early stopping) |
| `keypoints.py` | 3D Förstner + block matching |
| `masking.py` | LCC mask, structure masks |
| `tto.py` | `TTOConfig`, `run_tto`, `evaluate_registration` |
| `postprocessing.py` | mass-preserving HU correction |
| `dicom_export.py` | CT DICOM export from template and RTSTRUCT |
| `profiling.py` | VRAM profiling: per-block memory, inference and TTO peaks, per-phase TTO peaks, memory snapshot |
| `visualization.py` | all plots |
| `dose_analysis/` | planning structures (`build_structures.py`) and dose, gamma, geometry and DVH comparison (`dose_analysis.py`) |

### Paths
Every input and output location is set here. Paths are relative to the repository root (`REPO_DIR`).

| Variable | Content |
|---|---|
| `DATA_DIR` | Learn2Reg lung CT data (README, *Data*): `training/scans` and `training/lungMasks` (`case_xxx_exp` / `case_xxx_insp` images of the Learn2Reg 2020 training release) and, from the LungCT release, `landmarksTr`, `keypointsTr`, `imagesTs`, `masksTs`, `keypointsTs` |
| `CLINICAL_CASE` | name of the 4D-Lung case of sections 9-10 |
| `CLINICAL_DIR` | 4D-Lung case: `images/` and `masks/` with `<CLINICAL_CASE>_0000.nii.gz` (fixed) and `<CLINICAL_CASE>_0001.nii.gz` (moving), `real_ct_dicom/` (DICOM series of the real CT of the day), `baseline/` and `real_ct/` (CT slices + RTSTRUCT of the moving CT and of the real CT) |
| `WEIGHTS_PATH` | trained weights (`weights/README.md`); written by section 4 |
| `RESULTS_DIR` | text outputs of sections 4-10 (`results/`) |
| `OUTPUT_DIR` | everything else the notebook writes: TensorBoard runs, cache, keypoints, DVFs, DICOM / RTSTRUCT exports |


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = f"{REPO_DIR}/data/LungCT"
CLINICAL_CASE = "4dlung_107"
CLINICAL_DIR = f"{REPO_DIR}/data/clinical/{CLINICAL_CASE}"
OUTPUT_DIR = f"{REPO_DIR}/outputs"
RESULTS_DIR = f"{REPO_DIR}/results"
WEIGHTS_PATH = f"{REPO_DIR}/weights/best_reg_model.pth"

TRAIN_IMG_DIR = f"{DATA_DIR}/training/scans"
TRAIN_MSK_DIR = f"{DATA_DIR}/training/lungMasks"
TRAIN_LM_DIR = f"{DATA_DIR}/landmarksTr"
TRAIN_KP_DIR = f"{DATA_DIR}/keypointsTr"
TEST_IMG_DIR = f"{DATA_DIR}/imagesTs"
TEST_MSK_DIR = f"{DATA_DIR}/masksTs"
TEST_LM_DIR = f"{DATA_DIR}/landmarksTs"
TEST_KP_DIR = f"{DATA_DIR}/keypointsTs"

CLINICAL_IMG_DIR = f"{CLINICAL_DIR}/images"
CLINICAL_MSK_DIR = f"{CLINICAL_DIR}/masks"
CLINICAL_LM_DIR = f"{CLINICAL_DIR}/landmarks"
CLINICAL_KP_DIR = f"{CLINICAL_DIR}/keypoints"
REAL_CT_DICOM_DIR = f"{CLINICAL_DIR}/real_ct_dicom"
RTSTRUCT_MOV_DIR = f"{CLINICAL_DIR}/baseline"
RTSTRUCT_FIX_DIR = f"{CLINICAL_DIR}/real_ct"

RUNS_DIR = f"{OUTPUT_DIR}/runs"
LOCAL_DATA = f"{OUTPUT_DIR}/local_data"
CACHE_DIR = f"{LOCAL_DATA}/cache"
KP_OUT_DIR = f"{OUTPUT_DIR}/keypoints"
DVF_EXPORT_DIR = f"{OUTPUT_DIR}/dvf_exports"
WARPED_CT_EXPORT_DIR = f"{OUTPUT_DIR}/dir_cbct/ct"
RTSTRUCT_PATH = f"{OUTPUT_DIR}/dir_cbct/rtstruct/rtstruct_segmentations.dcm"
WARPED_CT_NOJAC_EXPORT_DIR = f"{OUTPUT_DIR}/dir_cbct_no_jacobian/ct"
RTSTRUCT_NOJAC_PATH = f"{OUTPUT_DIR}/dir_cbct_no_jacobian/rtstruct/rtstruct_segmentations.dcm"
SNAPSHOT_PATH = f"{OUTPUT_DIR}/vram_profiling/tto_memory_snapshot.pickle"

TRAINING_LOG = f"{RESULTS_DIR}/training/training_log.txt"
KEYPOINT_DIAGNOSTIC_LOG = f"{RESULTS_DIR}/training/keypoint_diagnostic.txt"
TTO_VALIDATION_LOG = f"{RESULTS_DIR}/tto/tto_validation_test.txt"
TTO_REGULARIZATION_LOG = f"{RESULTS_DIR}/tto/tto_regularization_study.txt"
KEYPOINT_EXTRACTION_LOG = f"{RESULTS_DIR}/keypoints/keypoint_extraction.txt"
VRAM_PROFILING_LOG = f"{RESULTS_DIR}/vram_profiling/vram_profiling.txt"
ATTENTION_QUALITY_LOG = f"{RESULTS_DIR}/vram_profiling/attention_quality.txt"


### 0. Environment
Create the environment and install the package from the repository root (README, *Installation*), then start TensorBoard with `tensorboard --logdir outputs/runs` to follow the training.

### 1. Package import
The cell removes the package from `sys.modules` before importing it (a replacement for `%autoreload`, which does not work on Python 3.13): after editing a `.py` file, re-running this cell reloads it without restarting the kernel. `tee_output(path)` prints as usual and also writes the output of a cell to `path`.

In [ ]:
import sys
import time
from contextlib import contextmanager, redirect_stdout

for module in [m for m in sys.modules if m == "registration" or m.startswith("registration.")]:
    del sys.modules[module]

import warnings
from dataclasses import replace

import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

from registration.augmentation import RegistrationAugmentation
from registration.dataset import InvertedPair, Learn2RegDataset, copy_to_local
from registration.dicom_export import (common_structures, create_rtstruct, export_dicom_from_template, is_bone,
                                       is_lung, read_reference_grid, read_rtstruct_masks, to_fixed_grid)
from registration.keypoints import extract_dataset_keypoints
from registration.losses import SuperRegistrationLoss, landmark_loss
from registration.masking import build_lcc_mask, lcc_mask_tensor, structures_to_tensor
from registration.metrics import binary_dice, compute_jacobian_determinant, jacobian_map
from registration.model import (SpatialTransformer3D, build_components, predict_dvf,
                                set_activation_checkpointing, set_window_attention)
from registration.postprocessing import mass_diagnostics, mass_preserving_hu_correction, warp_binary_mask
from registration.profiling import TTO_PHASES, catch_oom, memory_snapshot, profile_blocks, profile_tto
from registration.training import PhaseConfig, train_phase
from registration.tto import TTOConfig, evaluate_registration, run_tto
from registration.utils import (autocast_bf16, clear_vram, custom_collate, dvf_to_native_resolution, get_device,
                                get_warmup_cosine_scheduler, load_weights, network_spacing, read_nifti, set_seed,
                                volume_to_tensor, zero_init_output)
from registration.visualization import (central_slice, plot_ct_structures, plot_keypoint_coverage, plot_lcc_mask,
                                        plot_memory_profile, plot_tto_diagnostics, registration_viewer,
                                        structure_colors)

warnings.filterwarnings("ignore", category=FutureWarning, module="monai")


class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


@contextmanager
def tee_output(path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as log, redirect_stdout(_Tee(sys.stdout, log)):
        yield


def require(*names, hint):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not defined: {hint}")


device = get_device()
writer = SummaryWriter(RUNS_DIR)

### 2. Configuration
Flags and settings; the long stages only run if their flag is `True`.

- **Local data**: with `COPY_TRAINING_DATA = True` the training folders are copied once per session to `LOCAL_DATA`, useful when the data live on a slow network drive. The volumes already normalized and resampled to the network grid are also cached there (`CACHE_DIR`): the NIfTI files are decompressed and resampled only in the first epoch.
- **Point conventions**: the CSV files of landmarks and keypoints follow two different conventions, verified with the patch-correlation search. Training files have the x and y axes mirrored and the `_0000`/`_0001` phases swapped with respect to the `training/scans` images; test files follow the image convention. The mirroring is a property of the pair CSV / `case_xxx` images: with the `LungCT_xxxx` images of `imagesTr` the CSVs need no flip and no swap (`TEST_POINT_CONVENTION`).
- **Seed**: `SEED` fixes the model initialization, the batch order and the augmentation. The TTO stays non-deterministic on the GPU (section 10.1).
- **Augmentation**: done on the GPU in the training loop, not in the dataloader workers. The same rigid transform is applied to moving and fixed, so the augmentation does not add artificial relative motion to the breathing displacement the network has to learn.

In [ ]:
RUN_TRAINING = True
RUN_KEYPOINT_EXTRACTION = True
DIFFERENTIABLE_DICE = True
EXPORT_NO_JACOBIAN_CT = True
SEED = 42
set_seed(SEED)

COPY_TRAINING_DATA = False
if COPY_TRAINING_DATA:
    TRAIN_IMG_DIR = copy_to_local(TRAIN_IMG_DIR, "scans", LOCAL_DATA)
    TRAIN_MSK_DIR = copy_to_local(TRAIN_MSK_DIR, "lungMasks", LOCAL_DATA)
    TRAIN_LM_DIR = copy_to_local(TRAIN_LM_DIR, "landmarksTr", LOCAL_DATA)
    TRAIN_KP_DIR = copy_to_local(TRAIN_KP_DIR, "keypointsTr", LOCAL_DATA)
VAL_PATIENTS = ["0001", "0002", "0003"]

TRAIN_POINT_CONVENTION = dict(point_flip_axes=(0, 1), swap_point_phases=True)
TEST_POINT_CONVENTION = dict(point_flip_axes=(), swap_point_phases=False)

AUGMENTATION = dict(same_affine=True, max_rot=0.10, max_trans_vox=8.0)

FEATURE_SIZE = 48
INTEGRATION_STEPS = 9
COMPILE_MODEL = False  # torch.compile: optional, try it only if the GPU is the bottleneck
BATCH_SIZE = 1
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 2)

### 3. Dataset and model for training
Intra-patient pairs (expiration → inspiration) at 192³. Validation uses the `VAL_PATIENTS` without augmentation; training is doubled with the inverted pairs (moving and fixed swapped). The model is a SwinUNETR initialised with the MONAI self-supervised weights, with the last layer set to ~0 so that the initial DVF is the identity.

In [ ]:
TRAIN_TARGET_SHAPE = (192, 192, 192)

if RUN_TRAINING:
    train_dirs = (TRAIN_IMG_DIR, TRAIN_MSK_DIR, TRAIN_LM_DIR, TRAIN_KP_DIR)
    ds_val = Learn2RegDataset(*train_dirs, target_shape=TRAIN_TARGET_SHAPE, cache_dir=CACHE_DIR,
                              **TRAIN_POINT_CONVENTION)
    augmentation = RegistrationAugmentation(**AUGMENTATION)

    val_idx = [i for i, p in enumerate(ds_val.patient_per_pair) if p in VAL_PATIENTS]
    train_idx = [i for i, p in enumerate(ds_val.patient_per_pair) if p not in VAL_PATIENTS]
    train_fwd = Subset(ds_val, train_idx)
    train_dataset = ConcatDataset([train_fwd, InvertedPair(train_fwd)])
    val_dataset = Subset(ds_val, val_idx)

    loader_options = dict(pin_memory=True, collate_fn=custom_collate,
                          num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_options)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, **loader_options)
    print(f"Training pairs: {len(train_dataset)} | validation pairs: {len(val_dataset)}")

    model, warp_layer, integrator = build_components(device, TRAIN_TARGET_SHAPE, INTEGRATION_STEPS, FEATURE_SIZE,
                                                     compile_model=COMPILE_MODEL)
    zero_init_output(model)

### 4. Training: warmup + finetune
**Warmup** (30 epochs): the SwinUNETR transformer is frozen, everything else is trained, including the patch embedding. The self-supervised weights are identical on the moving and fixed channels, so a frozen patch embedding would only let the transformer see the mean of the two images.

**Finetune** (up to 300 epochs, early stopping with patience 40): the whole network starts from the best warmup model, with a differentiated learning rate (the pretrained transformer moves 5 times more slowly than the rest of the network).

Both phases use batch 1 with 6 accumulated steps (effective batch 6) and a linear curriculum on the smoothness, landmark and keypoint weights. The model with the best validation TRE (native grid, mm) is saved to `WEIGHTS_PATH`; a checkpoint (`resume_<PHASE>.pth` next to the weights, written every `checkpoint_every` epochs and whenever the best model improves) allows resuming after a disconnection. Both are written in the background, without pausing the training. To restart a phase from scratch, delete its checkpoint.

In [ ]:
with tee_output(TRAINING_LOG):
    if RUN_TRAINING:
        criterion = SuperRegistrationLoss(lambda_smooth=1.0, lambda_label=0.4,
                                          differentiable_dice=DIFFERENTIABLE_DICE).to(device)
        common = dict(model=model, warp_layer=warp_layer, integrator=integrator, criterion=criterion,
                      train_loader=train_loader, val_loader=val_loader, device=device,
                      weights_path=WEIGHTS_PATH, writer=writer, augmentation=augmentation)

        for p in model.parameters():
            p.requires_grad = True
        for p in model.backbone.swinViT.parameters():
            p.requires_grad = False
        for p in model.backbone.swinViT.patch_embed.parameters():
            p.requires_grad = True
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
        sched = get_warmup_cosine_scheduler(opt, warmup_epochs=3, total_epochs=30, min_lr_ratio=0.1)
        best_tre = train_phase(PhaseConfig("WARMUP", 30, acc_steps=6, patience=10,
                                           smooth_range=(1.0, 1.0), lm_range=(0.5, 0.5), kp_range=(0.5, 0.5)),
                               opt=opt, sched=sched, **common)

        load_weights(model, WEIGHTS_PATH, device)
        for p in model.parameters():
            p.requires_grad = True
        transformer = list(model.backbone.swinViT.parameters())
        transformer_ids = {id(p) for p in transformer}
        rest = [p for p in model.parameters() if id(p) not in transformer_ids]
        opt = torch.optim.AdamW([{"params": transformer, "lr": 2e-5},
                                 {"params": rest, "lr": 1e-4}],
                                weight_decay=1e-4)
        sched = get_warmup_cosine_scheduler(opt, warmup_epochs=5, total_epochs=300, min_lr_ratio=0.01)
        best_tre = train_phase(PhaseConfig("FINETUNE", 300, acc_steps=6, patience=40,
                                           smooth_range=(1.0, 1.5), lm_range=(0.5, 1.5), kp_range=(1.0, 2.0)),
                               opt=opt, sched=sched, best_val=best_tre, **common)
        print(f"Best validation TRE: {best_tre:.2f} mm")


### 5. Diagnostic: keypoint error on training and validation pairs
Mean keypoint error (mm, without augmentation) with the identity and with the best model, for every training and validation pair. Comparing the mean reduction of the two groups tells whether the network is limited by learning (both low) or by generalization (training high, validation low).

In [ ]:
require("ds_val", "train_idx", "val_idx", hint="run section 3 with RUN_TRAINING = True")
with tee_output(KEYPOINT_DIAGNOSTIC_LOG):
    load_weights(model, WEIGHTS_PATH, device)
    model.eval()
    with torch.no_grad():
        for group, indices in (("TRAINING", train_idx), ("VALIDATION", val_idx)):
            reductions = []
            print(f"\n{group}")
            for i in indices:
                item = custom_collate([ds_val[i]])
                if not item["has_keypoints"][0]:
                    continue
                mov, fix = item["moving"].to(device), item["fixed"].to(device)
                p_m, p_f = item["kp_moving"][0][None].to(device), item["kp_fixed"][0][None].to(device)
                sp = item["spacing_mm"][0].to(device)
                with autocast_bf16(device):
                    dvf, _ = predict_dvf(model, integrator, warp_layer, mov, fix)
                err_id = landmark_loss(torch.zeros_like(dvf.float()), p_m, p_f, sp).item()
                err_net = landmark_loss(dvf.float(), p_m, p_f, sp).item()
                reductions.append(100 * (1 - err_net / err_id))
                print(f"  {item['case_name'][0]:40s} identity {err_id:6.2f} mm -> network {err_net:6.2f} mm "
                      f"({reductions[-1]:4.0f}% lower)")
            print(f"  mean reduction: {np.mean(reductions):.0f}%")


### 6. TTO on the validation and test patients
For each patient: error without registration, with the network alone and with network + Test-Time Optimization (`EVAL_TTO_CFG`: 120 steps, global fit for 90 then local fit for 30, `lr=5e-3`, lung Dice weights (1.5, 1.0), `lambda_reg=0.5` chosen with section 7, backbone optimized together with the head, without LCC and tumor masks), plus the folding of the TTO DVF. The validation patients are evaluated with the TRE on the landmarks, which the TTO never uses. The Learn2Reg test patients have no landmarks, so they are evaluated with the error on the keypoints (mean distance in mm on the network grid, as in section 5); for them the keypoints are removed from the TTO input, so that the error stays an independent measure. `lambda_reg` is chosen on the validation patients only: the test keypoint error is an independent check of that choice. The test pairs are resampled to the training grid and cached like the training ones.

In [ ]:
require("ds_val", "val_idx", hint="run section 3 with RUN_TRAINING = True")
with tee_output(TTO_VALIDATION_LOG):
    EVAL_TTO_CFG = TTOConfig(n_steps=120, switch_step=90, lr=5e-3, freeze_backbone=False, lambda_lung=(1.5, 1.0),
                             lambda_reg=0.5)

    def keypoint_error(dvf, batch):
        return landmark_loss(dvf.float(), batch["kp_moving"][0][None].to(device), batch["kp_fixed"][0][None].to(device),
                             batch["spacing_mm"][0].to(device)).item()


    ds_test = Learn2RegDataset(TEST_IMG_DIR, TEST_MSK_DIR, TEST_LM_DIR, TEST_KP_DIR, target_shape=TRAIN_TARGET_SHAPE,
                               cache_dir=CACHE_DIR, **TEST_POINT_CONVENTION)
    cases = [("validation", ds_val, i) for i in val_idx] + [("test", ds_test, i) for i in range(len(ds_test))]
    results, tre_cases = [], []
    for group, dataset, i in cases:
        batch = custom_collate([dataset[i]])
        name = batch["case_name"][0]
        if batch["has_landmarks"][0]:
            tre_cases.append((group, dataset, i))
            lm_f, lm_m = batch["lm_fix_orig"][0].numpy(), batch["lm_mov_orig"][0].numpy()
            err_identity = np.linalg.norm((lm_f - lm_m) * batch["spacing_orig"][0].numpy(), axis=1).mean()
            metric, tto_batch = "TRE", batch
        elif batch["has_keypoints"][0]:
            err_identity = keypoint_error(torch.zeros((1, 3, *TRAIN_TARGET_SHAPE), device=device), batch)
            metric, tto_batch = "keypoint error", {**batch, "has_keypoints": [False]}
        else:
            print(f"\n--- {name} ({group}): no landmarks or keypoints, skipped ---")
            continue
        print(f"\n--- {name} ({group}) | {metric} without registration {err_identity:.2f} mm ---")

        load_weights(model, WEIGHTS_PATH, device)
        res = evaluate_registration(model, warp_layer, integrator, batch, device)
        err_net = res["tre"][0] if metric == "TRE" else keypoint_error(res["dvf"], batch)

        run_tto(model, warp_layer, integrator, tto_batch, device, WEIGHTS_PATH, EVAL_TTO_CFG)
        res = evaluate_registration(model, warp_layer, integrator, batch, device)
        err_tto = res["tre"][0] if metric == "TRE" else keypoint_error(res["dvf"], batch)
        fold = (jacobian_map(res["dvf"][0].cpu().numpy()) <= 0).mean() * 100
        if metric == "keypoint error":
            print(f"   keypoint error: network {err_net:.2f} mm -> network + TTO {err_tto:.2f} mm")
        results.append((group, metric, name, err_identity, err_net, err_tto, fold))

    for group, metric in (("validation", "TRE"), ("test", "keypoint error")):
        sel = [r for r in results if r[0] == group]
        if not sel:
            continue
        print(f"\n{group.upper()}: {metric} (mm)")
        print(f"{'Patient':40s} | no registration | network | network + TTO | TTO folding")
        for _, _, name, e0, e1, e2, fold in sel:
            print(f"{name:40s} | {e0:15.2f} | {e1:7.2f} | {e2:13.2f} | {fold:10.2f}%")
        means = [np.mean([r[k] for r in sel]) for k in (3, 4, 5, 6)]
        label = f"MEAN ({len(sel)} patients)"
        print(f"{label:40s} | {means[0]:15.2f} | {means[1]:7.2f} | {means[2]:13.2f} | {means[3]:10.2f}%")


### 7. TTO regularization study: TRE vs folding
The earlier 150-step TTO with `lambda_reg=0.01` reaches ~3 mm but leaves 3-9% of the voxels with $\det(J) \le 0$. The TTO of section 6 (`EVAL_TTO_CFG`, 60 steps, backbone optimized) is repeated on the validation and test patients with landmarks of section 6 with increasing smoothness weight, measuring TRE and folding both on the whole volume and inside the lungs, to find the weight that removes folding with the smallest TRE cost. The means are given per group: the weight should be chosen on the validation patients, so that the test patients stay an independent check.

In [ ]:
require("tre_cases", "EVAL_TTO_CFG", hint="run sections 3 and 6 first")
with tee_output(TTO_REGULARIZATION_LOG):
    def tto_eval(dataset, i, cfg):
        batch = custom_collate([dataset[i]])
        run_tto(model, warp_layer, integrator, batch, device, WEIGHTS_PATH, cfg)
        res = evaluate_registration(model, warp_layer, integrator, batch, device)
        det = jacobian_map(res["dvf"][0].cpu().numpy())
        lung = batch["label_fixed"][0, 0].numpy() > 0.5
        return batch["case_name"][0], res["tre"][0], (det <= 0).mean() * 100, (det[lung] <= 0).mean() * 100

    REG_WEIGHTS = (0.01, 0.5, 2.0)
    rows = []
    for lam in REG_WEIGHTS:
        for group, dataset, i in tre_cases:
            cfg = replace(EVAL_TTO_CFG, lambda_reg=lam, log_every=1000)
            rows.append((lam, group, *tto_eval(dataset, i, cfg)))

    print(f"\n{'lambda_reg':>10s} | {'group':10s} | {'patient':40s} | {'TRE':>6s} | {'fold all':>8s} | {'fold lung':>9s}")
    for lam, group, name, tre, f_all, f_lung in rows:
        print(f"{lam:10.2f} | {group:10s} | {name:40s} | {tre:6.2f} | {f_all:7.2f}% | {f_lung:8.2f}%")
    for lam in REG_WEIGHTS:
        for group in ("validation", "test"):
            sel = [r for r in rows if r[0] == lam and r[1] == group]
            if sel:
                label = f"MEAN ({len(sel)} patients)"
                print(f"{lam:10.2f} | {group:10s} | {label:40s} | {np.mean([r[3] for r in sel]):6.2f} | "
                      f"{np.mean([r[4] for r in sel]):7.2f}% | {np.mean([r[5] for r in sel]):8.2f}%")


### 8. Automatic keypoint extraction
Förstner corners on the moving image and NCC block matching on the fixed one, for the test pairs. Points are saved with the same file convention the dataset uses when reading them.

In [ ]:
with tee_output(KEYPOINT_EXTRACTION_LOG):
    if RUN_KEYPOINT_EXTRACTION:
        ds_kp = Learn2RegDataset(TEST_IMG_DIR, TEST_MSK_DIR, TEST_LM_DIR, KP_OUT_DIR,
                                 target_shape=TRAIN_TARGET_SHAPE, **TEST_POINT_CONVENTION)
        last_kp, last_moving = extract_dataset_keypoints(ds_kp, KP_OUT_DIR, device)
        if last_moving is not None:
            plot_keypoint_coverage(last_kp, last_moving)


### 9. TTO with RTSTRUCT structures and DICOM / RTSTRUCT export
TTO with `export_tto_cfg` (`freeze_backbone=False`: the whole network is optimized, as in the `full_network` run of `results/dosimetry/`; `freeze_backbone=True` optimizes the regression head only), on the clinical case `CLINICAL_CASE` (`CLINICAL_IMG_DIR`). The structures are read from the moving and fixed RTSTRUCTs and the ROIs with the same name in both are rasterized to binary masks: the union of the lung ROIs replaces the lung labels, the tumor keeps its own Dice (weight 4 in the global fit, 10 in the local fit), the bones (vertebrae, ribs, sternum, scapulae, clavicles) enter a separate mean Dice (weight 2) and every other common structure enters the mean OAR Dice (weight 1).

The body mask is built on the fixed image of the case (the CT or CBCT of the day, never the real CT in the CBCT case) and with `mask_dvf=False` it only confines the loss: the DVF is not zeroed outside the body, so there is no step on the skin surface. The mask is shown on the central slice of the fixed CT. The structure masks on the network grid are stored in bfloat16, exact for binary masks and half the memory of float32.

With `LOW_VRAM = True` the TTO uses the efficient window attention and activation checkpointing (section 10): the peak of the TTO drops by ~40% on this grid (section 10), each step takes ~30-35% longer, and the result is equivalent (section 10.1).

In [ ]:
EXPORT_TARGET_SHAPE = (448, 288, 32)
LOW_VRAM = False

ds_export = Learn2RegDataset(CLINICAL_IMG_DIR, CLINICAL_MSK_DIR, CLINICAL_LM_DIR, CLINICAL_KP_DIR,
                             target_shape=EXPORT_TARGET_SHAPE, **TEST_POINT_CONVENTION)
CASE_IDX = next(i for i, f in enumerate(ds_export.moving_cases) if os.path.basename(f).startswith(CLINICAL_CASE))
batch = custom_collate([ds_export[CASE_IDX]])
model, warp_layer, integrator = build_components(device, EXPORT_TARGET_SHAPE, INTEGRATION_STEPS, FEATURE_SIZE, load_ssl=False)
set_window_attention(model, efficient=LOW_VRAM)
set_activation_checkpointing(model, integrator, enabled=LOW_VRAM)
mov_ct_path, fix_ct_path = ds_export.pair_paths(CASE_IDX)
fix_ct_np = read_nifti(fix_ct_path)[0]
body_fix_np = build_lcc_mask(fix_ct_np, hu_threshold=-500)
plot_lcc_mask(fix_ct_np, body_fix_np)
lcc_mask = lcc_mask_tensor(body_fix_np, EXPORT_TARGET_SHAPE, device, mode="trilinear")

rt_mov, rt_colors = read_rtstruct_masks(RTSTRUCT_MOV_DIR)
rt_fix, _ = read_rtstruct_masks(RTSTRUCT_FIX_DIR)
tumor_name, oar_names = common_structures(rt_mov, rt_fix)
held_out = {"pulmonary vein", "brachiocephalic trunk", "right subclavian artery", "left subclavian artery",
            "left common carotid artery", "left brachiocephalic vein", "right brachiocephalic vein",
            "left atrial appendage", "superior vena cava", "left deep back muscle", "right deep back muscle",
            "costal cartilage"} | {f"right rib {i}" for i in range(1, 8)}
organ_names = [n for n in oar_names if not is_bone(n)]
bone_names = [n for n in oar_names if is_bone(n)]
loss_organs = [n for n in organ_names if n not in held_out]
heldout_names = [n for n in organ_names + bone_names if n in held_out]
loss_bones = [n for n in bone_names if n not in held_out]
structure_names = [tumor_name] + loss_organs + heldout_names + loss_bones
lung_names = [n for n in organ_names if is_lung(n)]
print(f"Tumor: {tumor_name} | {len(organ_names)} other organs: {', '.join(organ_names)}")
print(f"{len(bone_names)} bones: {', '.join(bone_names)}")

struct_mov = structures_to_tensor(rt_mov, structure_names, EXPORT_TARGET_SHAPE, device, dtype=torch.bfloat16)
struct_fix = structures_to_tensor(rt_fix, structure_names, EXPORT_TARGET_SHAPE, device, dtype=torch.bfloat16)
tumor_mov, tumor_fix = struct_mov[:, :1], struct_fix[:, :1]
n_organs = 1 + len(organ_names)
spacing = network_spacing(batch["spacing_orig"][0].numpy(), batch["shape_orig"][0].tolist(), EXPORT_TARGET_SHAPE)
tumor_np = tumor_fix[0, 0].float().cpu().numpy() > 0.5
tumor_region = ndi.distance_transform_edt(~tumor_np, sampling=spacing) <= 10.0
weight_region = torch.from_numpy(tumor_region).float().to(device)[None, None]
batch["label_moving"] = structures_to_tensor(rt_mov, lung_names, EXPORT_TARGET_SHAPE, device).amax(1, keepdim=True).cpu()
batch["label_fixed"] = structures_to_tensor(rt_fix, lung_names, EXPORT_TARGET_SHAPE, device).amax(1, keepdim=True).cpu()

organ_slice = slice(1, 1 + len(loss_organs))
heldout_slice = slice(organ_slice.stop, organ_slice.stop + len(heldout_names))
bone_slice = slice(heldout_slice.stop, len(structure_names))
print(f"Held out from the loss: {', '.join(heldout_names)}")

export_tto_cfg = TTOConfig(lr=1e-3, freeze_backbone=False, n_steps=100, switch_step=100,
                           lambda_lung=(2.0, 1.0), lambda_tumor=(4.0, 10.0), lambda_oar=4.0,
                           lambda_bone=2.0, mask_dvf=False)
dvf_mask = lcc_mask if export_tto_cfg.mask_dvf else None
export_tto_inputs = dict(lcc_mask=lcc_mask, weight_mask=weight_region, tumor_moving=tumor_mov, tumor_fixed=tumor_fix,
                         oar_moving=struct_mov[:, organ_slice], oar_fixed=struct_fix[:, organ_slice],
                         bone_moving=struct_mov[:, bone_slice], bone_fixed=struct_fix[:, bone_slice],
                         heldout_moving=struct_mov[:, heldout_slice], heldout_fixed=struct_fix[:, heldout_slice],
                         bone_mask=struct_fix[:, n_organs:].amax(1, keepdim=True) if bone_names else None)
clear_vram()
run_tto(model, warp_layer, integrator, batch, device, WEIGHTS_PATH, export_tto_cfg, callback=plot_tto_diagnostics,
        **export_tto_inputs)
result = evaluate_registration(model, warp_layer, integrator, batch, device, dvf_mask, dvf_dir=DVF_EXPORT_DIR)
final_dvf = result["dvf"]

#### 9.1 Warped structures and mass-preserving HU correction
The final DVF is brought to the native resolution of the CT, so that the common RTSTRUCT structures (with the Dice of each one before and after the registration), their lung union and the moving CT in HU are warped on the native grid instead of the network one. Inside the lung (tumor excluded) the density of the warped CT is corrected by $\det(J)$ for mass conservation. Outside the body of the fixed image the warped CT is set to air (−1000 HU) and that mask is exported as `BODY`, so the external contour matches the anatomy of the day; `mass_diagnostics` checks the density correction.

In [ ]:
native_shape = tuple(batch["shape_orig"][0].tolist())
native_dvf = dvf_to_native_resolution(final_dvf, native_shape)
native_warp = SpatialTransformer3D(native_shape).to(device)

warped_structures = {}
print(f"{'Structure':32s} | Dice before | Dice after")
for n in structure_names:
    mov_n = structures_to_tensor(rt_mov, [n], native_shape, device)
    fix_n = structures_to_tensor(rt_fix, [n], native_shape, device)[0, 0].cpu().numpy()
    warped_structures[n] = warp_binary_mask(mov_n, native_dvf, native_warp) > 0.5
    print(f"{n:32s} | {binary_dice(mov_n[0, 0].cpu().numpy(), fix_n):11.3f} | {binary_dice(warped_structures[n], fix_n):10.3f}")
lung_mov = structures_to_tensor(rt_mov, lung_names, native_shape, device).amax(1, keepdim=True)
lung_warp = warp_binary_mask(lung_mov, native_dvf, native_warp) > 0.5
tumor_warp = warped_structures[tumor_name]

hu_mov = volume_to_tensor(read_nifti(mov_ct_path)[0], native_shape, device)
with torch.no_grad():
    hu_warped = native_warp(hu_mov, native_dvf, mode="bilinear")
hu_corrected, _ = mass_preserving_hu_correction(hu_warped, native_dvf, lung_warp, tumor_warp)
body_fix = lcc_mask_tensor(body_fix_np, native_shape, device) > 0.5
air = torch.full_like(hu_warped, -1000.0)
hu_corrected = torch.where(body_fix, hu_corrected, air)
warped_structures["BODY"] = body_fix[0, 0].cpu().numpy()
hu_fix = volume_to_tensor(read_nifti(fix_ct_path)[0], native_shape, device)
lung_fix = structures_to_tensor(rt_fix, lung_names, native_shape, device).amax(1, keepdim=True)
mass_diagnostics(hu_mov, hu_warped, hu_corrected, native_dvf, lung_mov[0, 0].cpu().numpy(),
                 lung_warp, tumor_warp, hu_fix, lung_fix[0, 0].cpu().numpy())
warped_ct_np = hu_corrected[0, 0].cpu().numpy()
warped_ct_nojac_np = torch.where(body_fix, hu_warped, air)[0, 0].cpu().numpy()

#### 9.2 DICOM and RTSTRUCT export
The warped CT and the structures, already at native resolution, are brought onto the grid of the real CT (no interpolation when the shapes match); the CT is exported as a DICOM series copying the geometric tags of the real CT, and all the warped common structures as an RTSTRUCT referencing it, with the names and colours of the moving RTSTRUCT. With `EXPORT_NO_JACOBIAN_CT = True` the same CT without the $\det(J)$ correction is also exported, with its own RTSTRUCT, to compare the dose computed with and without the correction.

In [ ]:
grid = read_reference_grid(REAL_CT_DICOM_DIR, nifti_fallback=fix_ct_path)
fixed_shape = grid["shape"]
segmentations = {n: to_fixed_grid(m, fixed_shape, is_mask=True) for n, m in warped_structures.items()}
export_dicom_from_template(to_fixed_grid(warped_ct_np, fixed_shape), WARPED_CT_EXPORT_DIR, REAL_CT_DICOM_DIR)
create_rtstruct(segmentations, RTSTRUCT_PATH, WARPED_CT_EXPORT_DIR, colors=rt_colors)
if EXPORT_NO_JACOBIAN_CT:
    export_dicom_from_template(to_fixed_grid(warped_ct_nojac_np, fixed_shape), WARPED_CT_NOJAC_EXPORT_DIR,
                               REAL_CT_DICOM_DIR, series_name="CT_Warped_TTO_noJac")
    create_rtstruct(segmentations, RTSTRUCT_NOJAC_PATH, WARPED_CT_NOJAC_EXPORT_DIR, colors=rt_colors)

#### 9.3 Visual check
Interactive viewer of the registration with the moving, fixed and warped contours of every common structure, and warped CT with the contours of the exported structures.

In [ ]:
colors = structure_colors(structure_names, rt_colors)
with torch.no_grad():
    struct_warp = warp_layer(struct_mov, final_dvf, mode="bilinear") > 0.5
registration_viewer(result, batch, structures=[(n, colors[n], struct_mov[0, k], struct_fix[0, k], struct_warp[0, k])
                                               for k, n in enumerate(structure_names)])
plot_ct_structures(warped_ct_np, [
    (f"{n} (warped)", warped_structures[n], colors[n], 2.5 if n == tumor_name else 1.5,
     "--" if n == tumor_name else "-") for n in structure_names
], z=central_slice(tumor_warp, warped_ct_np.shape[2] // 2))

### 10. VRAM profiling (section 9)
Where the GPU memory of section 9 goes, before choosing how to reduce it. The section runs right after section 9 and reuses everything it loaded: model, case on the 448×288×32 grid, RTSTRUCT structures, body mask, tumor weight region, TTO configuration and inputs (`export_tto_inputs`). It measures a no_grad forward (inference), a forward with gradients followed by a backward, the section 9 TTO (2 steps, identical inputs and configuration) and the same TTO with the frozen backbone. Hooks on the top-level SwinUNETR blocks give, for each block, its weights and the memory it keeps alive: only the output in inference, the activations saved for the backward with gradients. The TTO peak is split into weights, gradients, Adam states and activations, and is also given per phase: network forward (input to integrated DVF), losses (forward and backward down to the DVF), network backward, optimizer step.

Everything is measured for four setups: MONAI window attention, which materialises the fp32 attention matrices of all windows, efficient attention (`set_window_attention`: `scaled_dot_product_attention` on groups of windows, with the shifted-window mask kept as one region index per token instead of a float32 matrix for every window; same parameters and same result up to bf16 rounding), and both again with activation checkpointing (`set_activation_checkpointing`: `encoder1`, `decoder2`, `decoder1` and every squaring step of the integrator keep only their inputs in the forward and are recomputed in the backward; same result, only with gradients). The maximum velocity difference from the first setup is printed as a check. At the end the setup chosen with `LOW_VRAM` in section 9 is restored.

The absolute peaks include every tensor already on the GPU (structures, masks, results of section 9): they are the real usage of section 9, not of the network alone. The model is reloaded with the best weights at the start, so that both attentions see the same weights: `final_dvf` and the results of section 9 are kept, but re-running section 9 cells that use `model` needs the TTO again. A configuration that does not fit is reported as `[OOM]` and skipped.

In [ ]:
require("export_tto_cfg", "export_tto_inputs", hint="run section 9 first, section 10 reuses its model, case and TTO inputs")
with tee_output(VRAM_PROFILING_LOG):
    PROFILE_SETUPS = {"MONAI": dict(efficient=False, checkpointing=False),
                      "efficient": dict(efficient=True, checkpointing=False),
                      "MONAI + checkpointing": dict(efficient=False, checkpointing=True),
                      "efficient + checkpointing": dict(efficient=True, checkpointing=True)}
    PROFILE_TTO_CONFIGS = {"section 9 TTO": export_tto_cfg,
                           "section 9 TTO, frozen backbone": replace(export_tto_cfg, freeze_backbone=True)}

    load_weights(model, WEIGHTS_PATH, device)
    clear_vram()
    print(f"Grid {EXPORT_TARGET_SHAPE} | {struct_mov.shape[1]} structures (moving and fixed)")
    profile_input = torch.cat([batch["moving"], batch["fixed"]], dim=1).to(device)
    profile_rows = []
    reference_velocity = None
    for setup, options in PROFILE_SETUPS.items():
        print(f"\n===== {setup} =====")
        set_window_attention(model, efficient=options["efficient"])
        set_activation_checkpointing(model, integrator, enabled=options["checkpointing"])
        model.eval()
        with torch.no_grad(), autocast_bf16(device):
            velocity = model(profile_input).float()
        if reference_velocity is None:
            reference_velocity = velocity
        else:
            difference = (velocity - reference_velocity).abs().max().item()
            print(f"Max |velocity difference| from {next(iter(PROFILE_SETUPS))}: {difference:.2e} "
                  f"(max |velocity| {reference_velocity.abs().max().item():.2f})")
        del velocity

        blocks = {}
        for scenario, training in (("inference", False), ("forward + backward", True)):
            result = catch_oom(profile_blocks, model, warp_layer, integrator, batch, device, training)
            if result is not None:
                blocks[scenario], summary = result
            profile_rows.append({"setup": setup, "scenario": scenario, **(summary if result is not None else {})})
        for scenario, cfg in PROFILE_TTO_CONFIGS.items():
            print(f"\n{scenario}:")
            summary = catch_oom(profile_tto, model, warp_layer, integrator, batch, device, WEIGHTS_PATH, cfg,
                                **export_tto_inputs)
            profile_rows.append({"setup": setup, "scenario": scenario, **(summary or {})})

        for scenario, df in blocks.items():
            print(f"\nPer-block memory, {scenario}:")
            print(df.round(1).to_string(index=False))
        if "inference" in blocks:
            plot_memory_profile(blocks["inference"], blocks.get("forward + backward"),
                                title=f"VRAM per block - section 9 {EXPORT_TARGET_SHAPE}, {setup}")

    set_window_attention(model, efficient=LOW_VRAM)
    set_activation_checkpointing(model, integrator, enabled=LOW_VRAM)
    del profile_input, reference_velocity

    profile_table = pd.DataFrame(profile_rows)
    columns = ["setup", "scenario", "peak_MiB", "baseline_MiB", "weights_MiB", "gradients_MiB", "adam_MiB",
               "activations_MiB", "forward_retained_MiB", "forward_s", "backward_s", "step_s"]
    phase_columns = ["setup", "scenario"] + [f"{phase}_peak_MiB" for phase in TTO_PHASES]
    print("\nSummary (MiB, seconds; NaN = not measured or out of memory):")
    print(profile_table.reindex(columns=columns).round(2).to_string(index=False))
    print("\nTTO peak per phase (MiB):")
    print(profile_table.dropna(subset=["step_s"]).reindex(columns=phase_columns).round(1).to_string(index=False))


#### 10.1 Efficient attention: TTO quality check
The section 9 TTO (100 steps, same configuration and inputs) is repeated with the MONAI window attention and with the efficient one. The TTO is not deterministic on the GPU (non-deterministic kernels, e.g. the `grid_sample` backward, amplified by Adam over 100 steps), so identical runs already differ: the repeated runs measure this variability and the efficient attention is equivalent if it stays inside it. For each run: final loss, lung Dice, mean Dice of the tumor, of the organs and bones in the loss and of the held-out structures (network grid, structures warped with the final DVF), folding and time, then their mean ± std per attention. The Dice of each structure is given as mean ± std per attention with the difference between the two. The DVF difference in mm (mean on the whole volume and inside the lungs, maximum) is computed for every pair of runs and grouped into MONAI vs MONAI, efficient vs efficient and MONAI vs efficient: the last one should be of the same order as the first two.

The results of section 9 (`final_dvf`) are not changed. Activation checkpointing stays as chosen with `LOW_VRAM` (it does not change the result). At the end the attention chosen with `LOW_VRAM` is restored and the model keeps the weights of the last run.

In [ ]:
require("export_tto_cfg", "export_tto_inputs", hint="run section 9 first, section 10 reuses its model, case and TTO inputs")
with tee_output(ATTENTION_QUALITY_LOG):
    QUALITY_RUNS = ("MONAI",) * 10 + ("efficient",) * 10

    structure_groups = {"tumor": [tumor_name], "organs (loss)": loss_organs, "held-out": heldout_names,
                        "bones (loss)": loss_bones}
    fixed_structures = struct_fix[0].float().cpu().numpy()
    quality_rows, quality_dice, quality_dvfs = [], [], []
    for k_run, attention in enumerate(QUALITY_RUNS, start=1):
        print(f"\n--- Run {k_run}/{len(QUALITY_RUNS)}: section 9 TTO, {attention} window attention ---")
        set_window_attention(model, efficient=attention == "efficient")
        clear_vram()
        t0 = time.perf_counter()
        history = run_tto(model, warp_layer, integrator, batch, device, WEIGHTS_PATH, export_tto_cfg, **export_tto_inputs)
        elapsed = time.perf_counter() - t0
        run = evaluate_registration(model, warp_layer, integrator, batch, device, dvf_mask)
        with torch.no_grad():
            warped = (warp_layer(struct_mov, run["dvf"], mode="bilinear") > 0.5)[0].cpu().numpy()
        dice = {n: binary_dice(warped[k], fixed_structures[k]) for k, n in enumerate(structure_names)}
        quality_rows.append({"attention": attention, "final loss": history[-1], "lung Dice": run["dice"][0],
                             **{f"{group} Dice": np.mean([dice[n] for n in names])
                                for group, names in structure_groups.items() if names},
                             "folding %": compute_jacobian_determinant(run["dvf"][0])[1], "time_s": elapsed})
        quality_dice.append(dice)
        quality_dvfs.append(run["dvf"][0].cpu())
        del run, warped
    set_window_attention(model, efficient=LOW_VRAM)

    dice_table = pd.DataFrame(quality_dice)
    dice_stats = dice_table.groupby(pd.Series(QUALITY_RUNS)).agg(["mean", "std"])
    print(f"\n{'Structure':32s} | {'MONAI':>13s} | {'efficient':>13s} | difference")
    for n in structure_names:
        m, e = dice_stats.loc["MONAI", n], dice_stats.loc["efficient", n]
        print(f"{n:32s} | {m['mean']:.3f} ± {m['std']:.3f} | {e['mean']:.3f} ± {e['std']:.3f} | "
              f"{e['mean'] - m['mean']:+.3f}")

    quality_table = pd.DataFrame(quality_rows)
    print("\nPer run:")
    print(quality_table.round(4).to_string())
    print("\nMean ± std per attention:")
    print(quality_table.groupby("attention").agg(["mean", "std"]).T.round(4).to_string())

    spacing_mm = torch.tensor(spacing, dtype=torch.float32).view(3, 1, 1, 1)
    lung = batch["label_fixed"][0, 0] > 0.5
    pair_rows = []
    for i in range(len(QUALITY_RUNS)):
        for j in range(i + 1, len(QUALITY_RUNS)):
            difference = ((quality_dvfs[i] - quality_dvfs[j]) * spacing_mm).norm(dim=0)
            pair_rows.append({"pair": " vs ".join(sorted((QUALITY_RUNS[i], QUALITY_RUNS[j]))),
                              "mean |dDVF| mm": difference.mean().item(),
                              "mean |dDVF| lung mm": difference[lung].mean().item(),
                              "max |dDVF| mm": difference.max().item()})
    print("\nDVF difference between pairs of runs (mean over the pairs of each group):")
    print(pd.DataFrame(pair_rows).groupby("pair").mean().round(3).to_string())


#### 10.2 Memory snapshot of the section 9 TTO
Every CUDA allocation, with the Python stack that made it, is recorded during a 2-step section 9 TTO with the efficient attention and activation checkpointing (the setup with the lowest peak) and saved as `SNAPSHOT_PATH`. Dropping the file onto https://pytorch.org/memory_viz shows the memory over time and which tensors make up the peak. The model is reloaded with the best weights at the end and the setup chosen with `LOW_VRAM` is restored.

In [ ]:
require("export_tto_cfg", "export_tto_inputs", hint="run section 9 first, section 10 reuses its model, case and TTO inputs")
os.makedirs(os.path.dirname(SNAPSHOT_PATH), exist_ok=True)
set_window_attention(model, efficient=True)
set_activation_checkpointing(model, integrator, enabled=True)
clear_vram()
summary = memory_snapshot(SNAPSHOT_PATH, profile_tto, model, warp_layer, integrator, batch, device, WEIGHTS_PATH,
                          export_tto_cfg, **export_tto_inputs)
set_window_attention(model, efficient=LOW_VRAM)
set_activation_checkpointing(model, integrator, enabled=LOW_VRAM)
print(f"Snapshot saved to {SNAPSHOT_PATH} ({os.path.getsize(SNAPSHOT_PATH) / 1024**2:.1f} MiB)")
print(f"Peak {summary['peak_MiB']:.0f} MiB | baseline {summary['baseline_MiB']:.0f} MiB | "
      + " | ".join(f"{phase} {summary[f'{phase}_peak_MiB']:.0f}" for phase in TTO_PHASES))